# Neural Network Loan Approval Classification

Train a feed-forward neural network using one of two data preparation paths:

1. load the raw SMOTE training and raw test files, then normalize and encode them inside the neural-network pipeline; or
2. load the already normalized files generated by `07_smote_NN.ipynb`.

Change only `DATA_MODE` in the first code cell to switch between the paths. Hyperparameters are selected using randomized search with 5-fold stratified cross-validation.

# Initialization and Data Preparation

### Load data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

seed = 1
np.random.seed(seed)

PROJECT_ROOT = Path("..")
INPUT_DIR = PROJECT_ROOT / "data" / "pre-processed"
OUTPUT_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose one option:
# "raw_smote"          -> raw SMOTE train + raw test; preprocessing is done in this notebook
# "processed_smote"    -> normalized SMOTE train + normalized test generated by notebook 07
# "processed_no_smote" -> normalized original train + normalized test generated by notebook 07
DATA_MODE = "raw_smote"

if DATA_MODE == "raw_smote":
    train_file = "loan_data_smoted_raw_train_4.csv"
    test_file = "loan_data_raw_test_1.csv"
    model_path = OUTPUT_DIR / "neural_network_raw_smote.joblib"
    needs_preprocessing = True
elif DATA_MODE == "processed_smote":
    train_file = "loan_data_nn_normal_train_smote.csv"
    test_file = "loan_data_nn_normal_test.csv"
    model_path = OUTPUT_DIR / "neural_network_smote.joblib"
    needs_preprocessing = False
elif DATA_MODE == "processed_no_smote":
    train_file = "loan_data_nn_normal_train.csv"
    test_file = "loan_data_nn_normal_test.csv"
    model_path = OUTPUT_DIR / "neural_network.joblib"
    needs_preprocessing = False
else:
    raise ValueError("DATA_MODE must be 'raw_smote', 'processed_smote', or 'processed_no_smote'.")

train_df = pd.read_csv(INPUT_DIR / train_file)
test_df = pd.read_csv(INPUT_DIR / test_file)

if set(train_df.columns) != set(test_df.columns):
    raise ValueError("The training and test files do not contain the same columns.")

test_df = test_df[train_df.columns]

print("Data mode:", DATA_MODE)
print("Training file:", train_file)
print("Test file:", test_file)
print("NN train data shape:", train_df.shape)
print("NN test data shape:", test_df.shape)

train_df.head()

### Data Checks

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()
print("Train columns:", train_df.columns.tolist())
print()
print("Missing values in training data:", train_df.isna().sum().sum())
print("Missing values in test data:", test_df.isna().sum().sum())
print("Train duplicate rows:", train_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

print()
print("Training target distribution:")
print(train_df["loan_status"].value_counts())
print()
print("Test target distribution:")
print(test_df["loan_status"].value_counts())

if needs_preprocessing:
    print()
    print("Raw categorical columns:", train_df.select_dtypes(include="object").columns.tolist())

### Create Training and Testing Data Matrices and Response Vectors

For `raw_smote`, continuous features are standardized and categorical features are one-hot encoded inside the model pipeline. `person_education` remains an ordinal value from 1 to 5, matching notebook 07.

For either processed mode, the files from notebook 07 are already normalized and encoded, so they are passed directly to the neural network.

In [ ]:
response_col = "loan_status"

X_train = train_df.drop(columns=[response_col])
y_train = train_df[response_col]

X_test = test_df.drop(columns=[response_col])
y_test = test_df[response_col]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

if needs_preprocessing:
    print("Raw data will be normalized and encoded inside the pipeline.")
else:
    print("All predictors numeric:", X_train.select_dtypes(exclude=np.number).empty)

# Neural Network Implementation

### Neural Network Tuning

In [ ]:
from scipy.stats import loguniform
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

nn_model = MLPClassifier(
    solver="adam",
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=12,
    random_state=seed,
)

if needs_preprocessing:
    education_col = "person_education"
    categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
    continuous_cols = X_train.select_dtypes(exclude="object").columns.tolist()
    continuous_cols.remove(education_col)

    preprocessor = ColumnTransformer([
        ("continuous", StandardScaler(), continuous_cols),
        ("education", "passthrough", [education_col]),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ], sparse_threshold=0)

    nn_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", nn_model),
    ])

    print("Continuous columns normalized:", continuous_cols)
    print("Ordinal column kept as 1-5:", education_col)
    print("Categorical columns one-hot encoded:", categorical_cols)
else:
    nn_pipeline = Pipeline([
        ("model", nn_model),
    ])

param_distributions = {
    "model__hidden_layer_sizes": [(32,), (64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
    "model__activation": ["relu", "tanh"],
    "model__alpha": loguniform(1e-5, 1e-2),
    "model__learning_rate_init": loguniform(1e-4, 5e-3),
    "model__batch_size": [64, 128, 256],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

nn_search = RandomizedSearchCV(
    estimator=nn_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=seed,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

nn_search.fit(X_train, y_train)

### Best Hyperparameters and Cross-Validation Results

In [ ]:
print("Best CV F1 score:", round(nn_search.best_score_, 4))
print("Best parameters:")
for name, value in nn_search.best_params_.items():
    print(f"{name}: {value}")

search_results = pd.DataFrame(nn_search.cv_results_).sort_values("rank_test_score")
search_results[["rank_test_score", "mean_train_score", "mean_test_score", "std_test_score", "mean_fit_time", "params"]].head(10)

### Test Set Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

best_nn_model = nn_search.best_estimator_
class_labels = best_nn_model.named_steps["model"].classes_
y_proba = best_nn_model.predict_proba(X_test)[:, 1]
y_pred = best_nn_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
by_class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Test precision:", round(precision_score(y_test, y_pred), 4))
print("Test recall:", round(recall_score(y_test, y_pred), 4))
print("Test F1 score:", round(f1_score(y_test, y_pred), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("Test PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

print()
print("Confusion matrix: Row = actual class, Column = predicted class")
print(cm)
print()
print("By-class accuracy:")
for class_label, accuracy in zip(class_labels, by_class_accuracy):
    print(f"Class {class_label}: {accuracy:.4f}")

print()
print("Classification report:")
print(classification_report(y_test, y_pred))

### Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels).plot(values_format="d")
plt.title("Neural Network Confusion Matrix")
plt.tight_layout()
plt.show()

### ROC Curve

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Neural Network ROC Curve")
plt.tight_layout()
plt.show()

### Precision-Recall Curve

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title("Neural Network Precision-Recall Curve")
plt.tight_layout()
plt.show()

### Training History of the Best Model

In [ ]:
best_mlp = best_nn_model.named_steps["model"]

plt.figure(figsize=(7, 4))
plt.plot(best_mlp.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Best Neural Network Training Loss")
plt.tight_layout()
plt.show()

if best_mlp.validation_scores_ is not None:
    plt.figure(figsize=(7, 4))
    plt.plot(best_mlp.validation_scores_)
    plt.xlabel("Iteration")
    plt.ylabel("Validation Accuracy")
    plt.title("Best Neural Network Validation Accuracy")
    plt.tight_layout()
    plt.show()

print("Iterations used:", best_mlp.n_iter_)
print("Final training loss:", round(best_mlp.loss_, 6))

### Save the Tuned Model

In [ ]:
from joblib import dump

dump(best_nn_model, model_path)
print("Saved model to:", model_path)